# Task 2: gRNA Design for CRISPRi (CXCL11)

### Changelogs:
1. **Target Adjustment:** Adapted the pipeline from SERPINB2 to target **CXCL11**, which was significantly upregulated in COVID-19, thus requiring **CRISPRi** (Interference/Repression) to restore normal expression.
2. **Biological Padding:** Target the first exon, but include a ±25bp biological padding to allow for proper protospacer overlap at the exon edges.
3. **Multiplexing:** Select a pool of the top n guides rather than a single guide, as CRISPR systems are highly synergistic when multiplexed.
4. **Resilient Filtering:** Use a weighted composite score (efficiency + specificity) to rank guides reasonably.


In [6]:
import gzip
import os

import matplotlib.pyplot as plt
import pandas as pd
import requests
from IPython.display import display

# Ensure output directories exist
os.makedirs('../data', exist_ok=True)
os.makedirs('../results/tables', exist_ok=True)

## 1. Target Definition (Task 1 Consensus)

Based on the `consensus_de_genes_strict.csv` output from Task 1, **CXCL11** is significantly upregulated in SARS-CoV-2 infection (LFC: +5.79).

Task 2 requires designing a guide RNA for **CRISPRi (Interference)** for this gene, as it must be **repressed** so that the CRISPRi system can restore its expression closer to baseline levels.


In [7]:
target_gene = 'CXCL11'
target_chrom = '4'
chrom_accession = 'NC_000004.12'
transcript_id = 'NM_005409.4' # Main transcript variant for CXCL11

print(f"Targeting: {target_gene} (Chromosome {target_chrom})")

Targeting: CXCL11 (Chromosome 4)


## 2. Sequence Extraction (First Exon + Padding)

To ensure our 20nt guide RNAs can properly overlap the edges of the first exon without falling outside the search space, we include a ±25bp flanking padding during sequence extraction.

In [8]:
gff_file = '../GCF_000001405.26_GRCh38_genomic.gff.gz'
target_exons = []

# Parse the GFF file to locate the exact coordinates of the target gene's exons
with gzip.open(gff_file, 'rt') as file:
    for line in file:
        if line.startswith('#'):
            continue
            
        if '\texon\t' in line and f'gene={target_gene};' in line and transcript_id in line:
            parts = line.strip().split('\t')
            if parts[0] == chrom_accession:
                target_exons.append({
                    'start': int(parts[3]), 
                    'end': int(parts[4]), 
                    'strand': parts[6]
                })

exons_df = pd.DataFrame(target_exons).drop_duplicates().reset_index(drop=True)

if not exons_df.empty:
    strand = exons_df['strand'].iloc[0]
    
    # Determine the first exon based on the coding strand
    if strand == '-':
        first_exon = exons_df.sort_values(by='end', ascending=False).iloc[0]
    else:
        first_exon = exons_df.sort_values(by='start', ascending=True).iloc[0]
        
    unpadded_start = first_exon['start']
    unpadded_end = first_exon['end']
        
    # Apply ±25bp flanking biological padding for the search space
    start_pos = unpadded_start - 25
    end_pos = unpadded_end + 25

    print(f"True First Exon located at: {unpadded_start} - {unpadded_end} (Strand {strand})")
    print(f"Extracting Search Space (with ±25bp padding): {start_pos} - {end_pos}")

    # Fetch the padded sequence from the Ensembl REST API
    server = "https://rest.ensembl.org"
    endpoint = f"/sequence/region/human/{target_chrom}:{start_pos}..{end_pos}:{1 if strand == '+' else -1}"
    
    response = requests.get(server + endpoint, headers={"Content-Type": "text/plain"})
    
    if response.ok:
        fasta_path = f"../data/{target_gene.lower()}_first_exon_padded.fasta"
        with open(fasta_path, 'w') as file:
            file.write(f">{target_gene}_first_exon_chr{target_chrom}_{start_pos}_{end_pos}\n")
            file.write(response.text)
            
        # Also fetch the STRICT unpadded sequence for verification later
        unpad_resp = requests.get(server + f"/sequence/region/human/{target_chrom}:{unpadded_start}..{unpadded_end}:{1 if strand == '+' else -1}", headers={"Content-Type": "text/plain"})
        unpadded_seq = unpad_resp.text        
        print(f"\n--- Unpadded First Exon Sequence ({unpadded_end - unpadded_start + 1} bp) ---")
        print(unpadded_seq)
        print(f"\nSuccess! Padded sequence saved to {fasta_path}")


True First Exon located at: 76035927 - 76036197 (Strand -)
Extracting Search Space (with ±25bp padding): 76035902 - 76036222


## 3. Guide Evaluation & Multiplex Pool Selection

**NOTE: Before running the next cells, please upload the padded fasta sequence to http://crispor.tefor.net/ and download the genome-wide evaluation results to `../data/crispor_cxcl11_results.xls`.**

### Tool Choice Justification (Why CRISPOR?)
As with the CRISPRa pipeline, we explicitly chose **CRISPOR** for this pipeline because it calculates the advanced **CFD (Cutting Frequency Determination) Specificity Score**.

In [9]:
# Load the external CRISPOR genome-wide evaluation results
results_file = f'../data/crispor_{target_gene.lower()}_results.xls'

if not os.path.exists(results_file):
    print(f"WARNING: {results_file} not found.")
    print("Please run the extracted sequence through CRISPOR and save the results file to proceed.")
else:
    guides_df = pd.read_excel(results_file, header=8) 
    
    col_map = {
        '#guideId': '#guideId', 
        'mitSpecScore': 'mitSpecScore', 
        'cfdSpecScore': 'cfdSpecScore', 
        "Doench '16-Score": 'Efficiency'
    }
    guides_df = guides_df.rename(columns=col_map)
    
    # Extract strand information from the guide ID
    guides_df['strand'] = guides_df['#guideId'].apply(lambda x: '+' if 'forw' in x else '-')
    
    # 1. Base Safety Filter (Industry Standard CFD > 50)
    mit_mask = guides_df['mitSpecScore'] > 50
    cfd_mask = guides_df['cfdSpecScore'] > 50
    safe_guides = guides_df[mit_mask & cfd_mask].copy()
    
    # 2. Strict Cut-Site Boundary Filter
    def rev_comp(seq):
        comp = {'A':'T', 'T':'A', 'C':'G', 'G':'C', 'N':'N'}
        return "".join(comp.get(b, 'N') for b in reversed(seq))
    
    with open(f"../data/{target_gene.lower()}_first_exon_padded.fasta", 'r') as f:
        padded_seq = "".join(l.strip() for l in f.readlines()[1:])
        
    padding_length = 25
    exon_length = len(padded_seq) - (2 * padding_length)
    
    def is_cut_inside_exon(row):
        seq = row['targetSeq']
        strand = row['strand']
        
        if strand == '+':
            idx = padded_seq.find(seq)
            if idx == -1: return False
            cut_site = idx + 17
        else:
            rev = rev_comp(seq)
            idx = padded_seq.find(rev)
            if idx == -1: return False
            cut_site = idx + 6
            
        return padding_length <= cut_site < (padding_length + exon_length)
    
    safe_guides['Valid_Cut_Site'] = safe_guides.apply(is_cut_inside_exon, axis=1)
    valid_guides = safe_guides[safe_guides['Valid_Cut_Site'] == True].copy()
    
    print(f"Of the {len(safe_guides)} safe guides, {len(valid_guides)} have their actual cut site inside the first exon.")
    
    # 3. Integrate manual CRISPOR UI Notes (SNPs, U6/U3, Efficiency, Enzymes)
    notes_file = f"../data/{target_gene.lower()}_crispor_notes.txt"
    if os.path.exists(notes_file):
        import re
        with open(notes_file, 'r') as fn:
            notes_text = fn.read()
        notes_parsed = []
        curr = {}
        for line in notes_text.split('\n'):
            line = line.strip()
            if not line: continue
            match = re.match(r'^\d+\.\s+([ACGT]+)\s+[ACGT]+$', line)
            if match:
                if curr: notes_parsed.append(curr)
                curr = {'targetSeq': match.group(1), 'Has_SNP': False, 'Poly_T': False, 'Ineff': False, 'Enzymes': False}
            elif line.startswith('.'):
                if any(c.isalpha() for c in line): curr['Has_SNP'] = True
            elif 'Not with U6' in line:
                curr['Poly_T'] = True
            elif 'Inefficient' in line:
                curr['Ineff'] = True
            elif 'Enzymes' in line:
                curr['Enzymes'] = True
        if curr: notes_parsed.append(curr)
        
        notes_df = pd.DataFrame(notes_parsed)
        valid_guides = valid_guides.merge(notes_df, on='targetSeq', how='left')
        
        # Strictly filter out flawed guides based on the notes
        valid_guides = valid_guides[valid_guides['Poly_T'] != True]
        valid_guides = valid_guides[valid_guides['Ineff'] != True]
        valid_guides = valid_guides[valid_guides['Has_SNP'] != True]
        valid_guides = valid_guides[valid_guides['Enzymes'] != True]
        print(f"After filtering by CRISPOR UI Notes, {len(valid_guides)} perfectly clean guides remain.")

    # 4. Composite Weighted Score (40% Efficiency, 60% Safety)
    valid_guides['Composite_Score'] = (valid_guides['Efficiency'] * 0.4) + (valid_guides['cfdSpecScore'] * 0.6)
    
    # 5. Final Selection (Top 3)
    top_pool = valid_guides.sort_values(by='Composite_Score', ascending=False).head(3)
    
    print(f"\nSelected optimal guides for CRISPRi Multiplex Pool:")
    display(top_pool[['#guideId', 'targetSeq', 'Efficiency', 'cfdSpecScore', 'Composite_Score']])
    
    out_csv = f'../results/tables/task2_multiplex_pool_{target_gene.lower()}.csv'
    top_pool.to_csv(out_csv, index=False)


Of the 10 safe guides, 10 have their actual cut site inside the first exon.
After filtering by CRISPOR UI Notes, 10 perfectly clean guides remain.

Selected optimal guides for CRISPRi Multiplex Pool:


,#guideId,targetSeq,Efficiency,cfdSpecScore,Composite_Score
6,76forw,GAAAGGTGCATGACTCAAAGAGG,65,83,75.8
3,264forw,GAAGGGCATGGCTATAGCCTTGG,54,89,75.0
4,295forw,TTGTGTGCTACAGTTGTTCAAGG,46,91,73.0


https://crispor.gi.ucsc.edu/crispor.py?batchId=1ne42O21Dq482je12Vcq